# Analyze Results

Jupyter notebook for visualization and analysis of benchmark results.

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# display plotly figures in notebook
pio.renderers.default = "notebook"

## Load Benchmark Results

Load the most recent benchmark results CSV file from the results directory.

In [ ]:
# "sum" or "latest"
# - "sum" will load the cumulative results file 'benchmark_results_sum.csv'
# - "latest" will load the most recent file in the results directory
file_mode = "sum"

In [ ]:
# Find the results file
results_dir = Path("../results")
csv_files = list(results_dir.glob("benchmark_results_*.csv"))

if not csv_files:
    display(Markdown("❌ **No results files found in ../results/**"))
    display(
        Markdown("Run the benchmark first: `uv run python scripts/run_benchmark.py`")
    )
    df = None
else:
    if file_mode == "latest":
        # Get most recent file
        results_file = max(csv_files, key=lambda p: p.stat().st_mtime)
    elif file_mode == "sum":
        results_file = results_dir / "benchmark_results_sum.csv"

    # Load data
    df = pd.read_csv(results_file)

    display(Markdown(f"✅ **Loaded {len(df)} results from `{results_file.name}`**"))

    # Display experiments
    if "Exp_Number" in df.columns:
        exp_count = df["Exp_Number"].nunique()
        exp_numbers = sorted(df["Exp_Number"].unique().tolist())
        display(
            Markdown(
                f"**Experiments**: {exp_count} ({', '.join(map(str, exp_numbers))})"
            )
        )

    methods = ", ".join(sorted(df["Method"].unique().tolist()))
    display(Markdown(f"**Methods**: {methods}"))
    status_counts = dict(df["Status"].value_counts())
    display(Markdown(f"**Status**: {status_counts}"))

In [ ]:
# sort df by Exp_Number and Method
if df is not None:
    df = df.sort_values(by=["Exp_Number", "Method"]).reset_index(drop=True)

# format recall, precision and F1 score to 3 decimals
for col in ["Recall", "Precision", "F1_Score", "Jaccard"]:
    df[col] = df[col].astype(float).round(3)

for col in ["Adjusted_SI", "Success_Index", "Runtime_Sec"]:
    df[col] = df[col].astype(float).round(1)

## Data Overview

Examine the structure and summary statistics of the benchmark results.

In [ ]:
identifiers = ["Exp_Number", "Experiment_ID", "Method", "Experiment_Description"]
# Metrics ordered by importance: F1 Score (most important), ASI, then others
metrics = [
    "F1_Score",
    "Adjusted_SI",
    "Recall",
    "Precision",
    "Success_Index",
    "Runtime_Sec",
]

exp_number2description = dict(
    set(zip(df["Exp_Number"], df["Experiment_Description"], strict=True))
)

In [ ]:
# Define consistent color palette for all methods
METHOD_COLORS = {
    "GEMSS": "#1f77b4",  # Blue
    "ALFESE_mrmr_tau0.5": "#ff7f0e",  # Orange
    "ALFESE_mrmr_tau1.0": "#d62728",  # Red
    "ALFESE_mi_tau0.5": "#2ca02c",  # Green
    "ALFESE_mi_tau1.0": "#9467bd",  # Purple
    "ALFESE_fcbf_tau0.5": "#8c564b",  # Brown
    "ALFESE_fcbf_tau1.0": "#e377c2",  # Pink
}

In [ ]:
# show everything in a single dataframe - for easy save to csv and later loading to latex
display(Markdown("## All Results"))
display(df[["Exp_Number", "Experiment_Description", "Method", "Status"] + metrics])

In [ ]:
# count failed and successful experiments per method
all_methods = sorted(df["Method"].unique())
success_counts = df[df["Status"] == "Success"].groupby("Method").size()
failed_counts = df[df["Status"] != "Success"].groupby("Method").size()

# Reindex to show all methods
success_counts = success_counts.reindex(all_methods, fill_value=0)
failed_counts = failed_counts.reindex(all_methods, fill_value=0)

# Combine into a dataframe
status_summary = pd.DataFrame(
    {"Successful_Count": success_counts, "Failed_Count": failed_counts}
)

display(Markdown("## Experiment Status Count by Method"))
display(status_summary)

In [ ]:
if df is not None:
    # Filter successful results globally
    df_success = df[df["Status"] == "Success"].copy()

    all_experiments = sorted(df["Exp_Number"].unique())

    for exp_number in all_experiments:
        df_exp = df[df["Exp_Number"] == exp_number]
        df_exp_success = df_exp[df_exp["Status"] == "Success"]

        # Get experiment ID for display
        exp_id = df_exp["Experiment_ID"].iloc[0] if len(df_exp) > 0 else "Unknown"

        display(Markdown(f"## 🔍 Experiment {exp_number}: `{exp_id}`"))
        display(Markdown(f"{exp_number2description.get(exp_number, '')}"))

        if df_exp_success.empty:
            display(Markdown(f"⚠️ **No successful runs for experiment {exp_number}**"))
        else:
            display(Markdown(f"### Performance Metrics Summary:"))
            stats = df_exp_success[["Method"] + metrics]
            display(stats)
else:
    display(Markdown("⚠️ **No data loaded**"))

## Performance Metrics Across Experiments

Visualize how each metric varies across experiments, comparing methods.

In [ ]:
if df is not None and df_success is not None and len(df_success) > 0:
    # Create a figure with subplots for each metric
    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=(
            "F1 Score",
            "Adjusted Success Index",
            "Recall",
            "Precision",
            "Success Index",
            "Runtime (seconds)",
        ),
        vertical_spacing=0.12,
        horizontal_spacing=0.1,
    )

    # Sort by experiment number for better visualization
    df_plot = df_success.sort_values("Exp_Number")

    # Plot each metric - prioritizing F1 Score and ASI
    metrics_info = [
        ("F1_Score", 1, 1, [0, 1.1], "linear"),
        ("Adjusted_SI", 1, 2, None, "log"),
        ("Recall", 2, 1, [0, 1.1], "linear"),
        ("Precision", 2, 2, [0, 1.1], "linear"),
        ("Success_Index", 3, 1, None, "linear"),
        ("Runtime_Sec", 3, 2, None, "log"),
    ]

    for metric, row, col, y_range, y_type in metrics_info:
        for method in df_plot["Method"].unique():
            method_data = df_plot[df_plot["Method"] == method]
            fig.add_trace(
                go.Scatter(
                    x=method_data["Exp_Number"],
                    y=method_data[metric],
                    mode="lines+markers",
                    name=method,
                    legendgroup=method,
                    showlegend=(row == 1 and col == 1),  # Only show legend once
                    marker=dict(size=8, color=METHOD_COLORS.get(method, "#17becf")),
                    line=dict(width=2, color=METHOD_COLORS.get(method, "#17becf")),
                ),
                row=row,
                col=col,
            )

        # Update axes
        fig.update_xaxes(title_text="Experiment Number", dtick=1, row=row, col=col)
        fig.update_yaxes(
            title_text=metric.replace("_", " "), type=y_type, row=row, col=col
        )
        if y_range:
            fig.update_yaxes(range=y_range, row=row, col=col)

    # Update layout
    fig.update_layout(
        height=1200,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        title_text="Performance Metrics Across All Experiments",
    )

    fig.show()
else:
    display(Markdown("⚠️ **No data available**"))

## Method Comparison by Experiment

Compare methods within each experiment

In [ ]:
if df is not None and df_success is not None and len(df_success) > 0:
    # Create pivot tables for F1 Score and Adjusted SI by experiment
    fig_comp = make_subplots(
        rows=4,
        cols=1,
        subplot_titles=(
            "F1 Score",
            "Adjusted SI",
            "Recall",
            "Precision",
        ),
        vertical_spacing=0.1,
    )

    # F1 Score by Experiment and Method
    for method in sorted(df_success["Method"].unique()):
        method_data = df_success[df_success["Method"] == method].sort_values(
            "Exp_Number"
        )
        fig_comp.add_trace(
            go.Bar(
                name=method,
                x=method_data["Exp_Number"],
                y=method_data["F1_Score"],
                marker_color=METHOD_COLORS.get(method, "#17becf"),
                legendgroup=method,
                showlegend=True,
            ),
            row=1,
            col=1,
        )

    # Adjusted SI by Experiment and Method
    for method in sorted(df_success["Method"].unique()):
        method_data = df_success[df_success["Method"] == method].sort_values(
            "Exp_Number"
        )
        fig_comp.add_trace(
            go.Bar(
                name=method,
                x=method_data["Exp_Number"],
                y=method_data["Adjusted_SI"],
                marker_color=METHOD_COLORS.get(method, "#17becf"),
                legendgroup=method,
                showlegend=False,
            ),
            row=2,
            col=1,
        )

    # Recall by Experiment and Method
    for method in sorted(df_success["Method"].unique()):
        method_data = df_success[df_success["Method"] == method].sort_values(
            "Exp_Number"
        )
        fig_comp.add_trace(
            go.Bar(
                name=method,
                x=method_data["Exp_Number"],
                y=method_data["Recall"],
                marker_color=METHOD_COLORS.get(method, "#17becf"),
                legendgroup=method,
                showlegend=False,
            ),
            row=3,
            col=1,
        )

    # Precision by Experiment and Method
    for method in sorted(df_success["Method"].unique()):
        method_data = df_success[df_success["Method"] == method].sort_values(
            "Exp_Number"
        )
        fig_comp.add_trace(
            go.Bar(
                name=method,
                x=method_data["Exp_Number"],
                y=method_data["Precision"],
                marker_color=METHOD_COLORS.get(method, "#17becf"),
                legendgroup=method,
                showlegend=False,
            ),
            row=4,
            col=1,
        )

    # Update layout
    fig_comp.update_xaxes(title_text="Experiment Number", row=1, col=1)
    fig_comp.update_xaxes(title_text="Experiment Number", row=2, col=1)
    fig_comp.update_xaxes(title_text="Experiment Number", row=3, col=1)
    fig_comp.update_xaxes(title_text="Experiment Number", row=4, col=1)
    fig_comp.update_yaxes(title_text="F1 Score", range=[0, 1.1], row=1, col=1)
    fig_comp.update_yaxes(
        title_text="Adjusted SI (log scale)", type="log", row=2, col=1
    )
    fig_comp.update_yaxes(title_text="Recall", range=[0, 1.1], row=3, col=1)
    fig_comp.update_yaxes(title_text="Precision", range=[0, 1.1], row=4, col=1)

    fig_comp.update_layout(
        height=1200,
        barmode="group",
        title_text="Method Performance Comparison by Experiment",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )

    fig_comp.show()
else:
    display(Markdown("⚠️ **No data available**"))

In [ ]:
## Feature Selection Quality
# Analyze the trade-off between correct features and false positives.


# if df is not None and df_success is not None and len(df_success) > 0:
#     # Create subplots
#     fig_quality = make_subplots(
#         rows=1,
#         cols=2,
#         subplot_titles=(
#             "Feature Selection Quality: Correct vs False Positives",
#             "Solutions Found by Experiment and Method",
#         ),
#     )

#     # Scatter: Correct vs Extra features (colored by experiment)
#     for method in df_success["Method"].unique():
#         method_data = df_success[df_success["Method"] == method]
#         fig_quality.add_trace(
#             go.Scatter(
#                 x=method_data["n_correct"],
#                 y=method_data["n_extra"],
#                 mode="markers",
#                 name=method,
#                 opacity=0.6,
#                 marker=dict(size=10, color=METHOD_COLORS.get(method, "#17becf")),
#             ),
#             row=1,
#             col=1,
#         )

#     # Solutions found by experiment and method (grouped bar chart)
#     for method in sorted(df_success["Method"].unique()):
#         method_data = df_success[df_success["Method"] == method].sort_values(
#             "Exp_Number"
#         )
#         fig_quality.add_trace(
#             go.Bar(
#                 name=method,
#                 x=method_data["Exp_Number"],
#                 y=method_data["Solutions_Found"],
#                 marker_color=METHOD_COLORS.get(method, "#17becf"),
#                 legendgroup=method,
#                 showlegend=False,
#             ),
#             row=1,
#             col=2,
#         )

#     # Update layout
#     fig_quality.update_xaxes(title_text="Correct Features (n_correct)", row=1, col=1)
#     fig_quality.update_yaxes(title_text="False Positives (n_extra)", row=1, col=1)
#     fig_quality.update_xaxes(title_text="Experiment Number", row=1, col=2)
#     fig_quality.update_yaxes(title_text="Number of Solutions", row=1, col=2)

#     fig_quality.update_layout(
#         height=600,
#         showlegend=True,
#         barmode="group",
#         title_text="Feature Selection Quality Analysis (Per-Experiment)",
#     )

#     fig_quality.show()
# else:
#     display(Markdown("⚠️ **No data available**"))

## Best Performers by Experiment

Identify the best method for each experiment based on F1 Score.

In [ ]:
if df is not None and df_success is not None and len(df_success) > 0:
    display(Markdown("### Best Method per Experiment (by F1 Score)"))

    # Find best method for each experiment
    best_per_exp = []
    for exp_num in sorted(df_success["Exp_Number"].unique()):
        exp_data = df_success[df_success["Exp_Number"] == exp_num]
        best_row = exp_data.loc[exp_data["F1_Score"].idxmax()]

        best_per_exp.append(
            {
                "Exp_Number": int(best_row["Exp_Number"]),
                "Experiment_ID": best_row["Experiment_ID"],
                "Best_Method": best_row["Method"],
                "F1_Score": round(best_row["F1_Score"], 3),
                "Adjusted_SI": round(best_row["Adjusted_SI"], 3),
                "Recall": round(best_row["Recall"], 3),
                "Precision": round(best_row["Precision"], 3),
                "Runtime_Sec": round(best_row["Runtime_Sec"], 1),
            }
        )

    best_df = pd.DataFrame(best_per_exp)
    display(best_df)

    # Overall method winning statistics
    display(Markdown("\n### Method Win Statistics (across all experiments, F1 Score)"))

    # Get all methods and their win counts (including 0s)
    # Reverse sort to show GEMSS first
    all_methods = sorted(df_success["Method"].unique(), reverse=True)
    method_wins = (
        best_df["Best_Method"].value_counts().reindex(all_methods, fill_value=0)
    )
    win_stats = pd.DataFrame(
        {
            "Method": method_wins.index,
            "Experiments_Won": method_wins.values,
            "Win_Rate": (method_wins.values / len(best_df) * 100).round(1),
        }
    )
    display(win_stats)

    # Visualize method wins and runtime comparison
    fig_comparison = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Experiments Won (F1 Score)",
            "Average Runtime by Method",
        ),
        horizontal_spacing=0.15,
    )

    # Left plot: Experiments won
    fig_comparison.add_trace(
        go.Bar(
            x=win_stats["Method"],
            y=win_stats["Experiments_Won"],
            marker_color=[METHOD_COLORS.get(m, "#17becf") for m in win_stats["Method"]],
            text=win_stats["Win_Rate"].apply(lambda x: f"{x}%"),
            textposition="outside",
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    # Right plot: Average runtime by method (keep same order as other plots)
    runtime_stats = (
        df_success.groupby("Method")["Runtime_Sec"]
        .mean()
        .reindex(all_methods)
        .reset_index()
    )

    fig_comparison.add_trace(
        go.Bar(
            x=runtime_stats["Method"],
            y=runtime_stats["Runtime_Sec"],
            marker_color=[
                METHOD_COLORS.get(m, "#17becf") for m in runtime_stats["Method"]
            ],
            text=runtime_stats["Runtime_Sec"].apply(lambda x: f"{x:.1f}s"),
            textposition="outside",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    # Update layout
    fig_comparison.update_xaxes(title_text="Method", row=1, col=1)
    fig_comparison.update_yaxes(title_text="No. Experiments Won", row=1, col=1)
    fig_comparison.update_xaxes(title_text="Method", row=1, col=2)
    fig_comparison.update_yaxes(title_text="Runtime (seconds)", row=1, col=2)

    fig_comparison.update_layout(
        height=500,
        title_text="Method Performance Comparison: F1 Score Wins vs Runtime",
        showlegend=False,
    )

    fig_comparison.show()

else:
    display(Markdown("⚠️ **No data available for analysis. Run benchmark first.**"))

---

## Notes

### Methodology:

This benchmark compares feature selection methods using **synthetically generated datasets with known ground truth**. 

**Why Synthetic Data?**
- **Known Ground Truth**: True feature sets are known by design, enabling precise evaluation
- **Controlled Properties**: Each dataset is generated with specific parameters (sample size, feature count, noise, missing data, class imbalance)
- **Reproducibility**: Seeded generation ensures consistent results across runs
- **Systematic Testing**: Multiple experiments across different tiers isolate specific challenges:
  - **Tier 1**: Basic validation on clean data
  - **Tier 2**: High-dimensional stress test (P ≥ 1000)
  - **Tier 4**: Robustness under adversity (high noise, missing data)
  - **Tier 7**: Class imbalance scenarios

The tiered approach is inherited from the validation of GEMSS and all the experiments are taken from the original test suite (except for the increase of minimal sample size $n=25$ to $n=30$ due to limitations of ALFESE.)

**Methods Compared:**
- **GEMSS**: Multi-solution feature selector using Bayesian framework
- **ALFESE** (multiple variants): Knockoff-based method with different feature scorers (mrmr, mi, fcbf) and thresholds.

**Evaluation Approach:**
- Each method returns one or more feature sets (candidate solutions)
- The **union** of all selected features is compared against the ground truth
- Metrics quantify recovery (Recall), accuracy (Precision), and overall quality (F1, Adjusted SI)

### Metrics Explained:

**Primary Metrics:**
- **F1 Score** (Most Important): Harmonic mean of precision and recall, balances both metrics (higher is better)
  $$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
- **Adjusted SI**: Success Index weighted by precision, combines recall and precision (higher is better)
  $$\text{Adjusted SI} = \text{SI} \cdot \text{Precision}$$

**Secondary Metrics:**
- **Recall**: Fraction of true features successfully recovered (higher is better)
  $$\text{Recall} = \frac{\text{correct features}}{\text{total true features}}$$
- **Precision**: Fraction of selected features that are true positives (higher is better)
  $$\text{Precision} = \frac{\text{correct features}}{\text{total selected features}}$$
- **Success Index (SI)**: GEMSS-specific metric balancing coverage and accuracy
  $$\text{SI} = \frac{P \cdot \text{correct}}{(\text{true features})^2}$$
- **Solutions Found**: Number of distinct feature sets discovered

**Runtime:**
- Execution time in seconds.
- For GEMSS, it is determined primarily by: number of iterations and optimization batch size.
- For ALFESE, it is most influenced by the number of a model's fittings. We use the default settings everywhere.

### Usage:
1. Run the benchmark: `uv run python scripts/run_benchmark.py`
2. Results are saved to: `results/benchmark_results_YYYYMMDD_HHMMSS.csv`
3. Run all cells in this notebook to generate visualizations
4. Export to HTML: `jupyter nbconvert --to html --no-input scripts/analyze_results.ipynb`

### Tips:
- Use `df_success` to filter only successful runs
- Use `df[df['Exp_Number'] == 1]` to analyze specific experiments
- All plots are interactive - hover for details, zoom, pan, and download
- Focus on F1 Score for overall method comparison
- Refer to `configs/benchmark_config.yaml` for experiment details